# Notebook 2: Deskriptive Statistik und Verteilungsdiagnostik

**Statistik für Data Science** · Kurs 3,906 · Bachelor Computer Science · HS2026
Universität St. Gallen

---
### Projektteam:
- Andrin Greitmann
- Noah Appenzeller
- Manuel Schmid

---

Dieses Notebook führt die Inhalte von **VL02 (Deskriptive Statistik und Verteilungsdiagnostik)** in Python aus. 

## 0. Setup
Importiert `pandas`, `numpy`, `seaborn` und `matplotlib` und setzt den Seed.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Seed: Startwert für den Zufallsgenerator. Gleicher Seed bei jedem Durchlauf damit Resultate deterministisch & reproduzierbar sind. 
# Den Wert selbst haben wir beliebig gewählt, wir haben ihn als erstes vor der Analyse gewählt und danach nicht mehr geändert.
# Der Seed wird für alles Zufällige in allen Notebooks verwendet.
SEED = 2001
RNG = np.random.default_rng(SEED)

# Plot-Defauls
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7.5, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

In [ ]:
import sys

print("Python  ", sys.version.split()[0])
for modul in (np, pd, sns):
    print(f"{modul.__name__:8s}", modul.__version__)
print("Seed    ", SEED)

### Unseren Datensatz laden

In [ ]:
from ucimlrepo import fetch_ucirepo

# Datensatz direkt aus dem UCI Repository laden (ID 117 = Census-Income KDD)
census_income_kdd = fetch_ucirepo(id=117)

# Etwas spezielles an unserem Datenset: ucimlrepo liefert die Daten aufgeteilt in Features und Target. 
# (Wahrscheinlich für Machine Learning Projekte/Algorithmen)
# Target ist das Einkommen (über/unter 50K), Features sind alle anderen Merkmale.
X = census_income_kdd.data.features
y = census_income_kdd.data.targets

# Für die EDA wollten wir alle Spalten in einem DataFrame damit das Einkommen mit den anderen Merkmalen verglichen werden kann.
# join() funktioniert hier zum Glück recht einfach, weil X und y dieselben Zeilen in derselben Reihenfolge haben.
census = X.join(y)
 
# Leerzeichen bei den Textspalten entfernen
text_spalten = census.select_dtypes(exclude="number").columns
census[text_spalten] = census[text_spalten].apply(lambda s: s.str.strip())

# Wir wollen die Spalten umbenennen, sodass es ersichtlicher ist, mit was wir arbeiten
# Dies ist jedoch in den Metadaten der folgenden Ausgabe noch nicht ersichtlich, erst bei allen späteren Ausgaben.

spalten_umbenennung = {
    "AAGE": "AGE", "ACLSWKR": "CLASS_OF_WORKER", "ADTINK": "INDUSTRY_CODE",      
    "ADTOCC": "OCCUPATION_CODE", "AHGA": "EDUCATION", "AHSCOL": "ENROLLED_IN_EDU",
    "AMARITL": "MARITAL_STATUS", "AMJIND": "MAJOR_INDUSTRY", "AMJOCC": "MAJOR_OCCUPATION",
    "ARACE": "RACE", "AREORGN": "HISPANIC_ORIGIN", "ASEX": "SEX",
    "AUNMEM": "LABOR_UNION_MEMBER", "AUNTYPE": "UNEMPLOYMENT_REASON", "AWKSTAT": "EMPLOYMENT_STATUS",
    "CAPGAIN": "CAPITAL_GAINS", "GAPLOSS": "CAPITAL_LOSSES", "DIVVAL": "DIVIDENDS",
    "FILESTAT": "TAX_FILER_STATUS", "GRINREG": "PREV_REGION", "GRINST": "PREV_STATE",
    "HHDFMX": "HOUSEHOLD_STAT_DETAILED", "HHDREL": "HOUSEHOLD_SUMMARY", "MARSUPWRT": "WEIGHT",
    "MIGMTR1": "MIGRATION_MSA", "MIGMTR3": "MIGRATION_REG", "MIGMTR4": "MIGRATION_WITHIN_REG",
    "MIGSAME": "SAME_HOUSE_1YR", "MIGSUN": "MIGRATION_SUNBELT", "NOEMP": "NUM_EMPLOYEES",
    "PARENT": "PARENTS_PRESENT", "PEFNTVTY": "BIRTH_COUNTRY_FATHER", "PEMNTVTY": "BIRTH_COUNTRY_MOTHER",
    "PENATVTY": "BIRTH_COUNTRY_SELF", "PRCITSHP": "CITIZENSHIP", "SEOTR": "SELF_EMPLOYED",
    "VETQVA": "VET_QUESTIONNAIRE", "VETYN": "VETERANS_BENEFITS", "WKSWORK": "WEEKS_WORKED",
    "AHRSPAY": "WAGE_PER_HOUR", "year": "YEAR", "income": "INCOME"
}
census = census.rename(columns=spalten_umbenennung)

# Metadaten: Herkunft, Beschreibung und Erhebung des Datensatzes (Damit wir auf Bias analysieren können)
print("Metadata: " + str(census_income_kdd.metadata))

print(f"Rows: {census.shape[0]}, Columns: {census.shape[1]}")

# Variablenbeschreibung: Name, Typ und Bedeutung jeder Spalte (Grundlage um später Messniveau zu bestimmen)
print("Variable descriptions:")

display(census_income_kdd.variables)

### Datensatz bereinigen
Hier wenden wir die Bereinigung aus NB01 (Abschnitt 5) an. Danach ist `census` der bereinigte Datensatz, die Rohdaten bleiben als `census_raw` verfügbar.

In [ ]:
# Die Abfrage sorgt dafür, dass ein zweites Ausführen dieser Zelle wieder
# bei den Rohdaten startet (sonst würde z.B. der Stundenlohn ein zweites Mal durch 100 geteilt).
if "census_raw" not in globals():
    census_raw = census
census = census_raw.copy()

# 1) Migrationsblock 1995 wurde nicht erhoben, auch wo "Not in universe" steht (3.2)
jahr_95 = census["YEAR"] == 95
census.loc[jahr_95, ["SAME_HOUSE_1YR", "PREV_REGION"]] = np.nan

# 2) Die Bundesstaaten passen nicht zur Region, die Spalte verwenden wir nicht (3.3)
census = census.drop(columns="PREV_STATE")

# 3) Stundenlohn: 0 bei Erwerbstätigen heisst "nicht erfasst" (3.2), die Einheit ist Cents (4.2)
nicht_erfasst = (census["WEEKS_WORKED"] > 0) & (census["WAGE_PER_HOUR"] == 0)
census["WAGE_PER_HOUR"] = census["WAGE_PER_HOUR"].astype(float).where(~nicht_erfasst) / 100

# 4) Antwortausfall bei der hispanischen Herkunft (3.2)
census["HISPANIC_ORIGIN"] = census["HISPANIC_ORIGIN"].replace({"Do not know": np.nan, "NA": np.nan})

# 5) Geburtsländer nicht imputieren, sondern als eigene Kategorie führen (3.4 und 3.7)
census["BIRTH_COUNTRY_SELF"] = census["BIRTH_COUNTRY_SELF"].fillna("Foreign, Unknown")
eltern = ["BIRTH_COUNTRY_FATHER", "BIRTH_COUNTRY_MOTHER"]
census[eltern] = census[eltern].fillna("Unknown")

# 6) Messniveau im Datentyp festhalten, damit pandas keine unzulässigen Mittelwerte rechnet (1 und 4)
BILDUNG = [
    "Children",
    "Less than 1st grade",
    "1st 2nd 3rd or 4th grade",
    "5th or 6th grade",
    "7th and 8th grade",
    "9th grade",
    "10th grade",
    "11th grade",
    "12th grade no diploma",
    "High school graduate",
    "Some college but no degree",
    "Associates degree-occup /vocational",
    "Associates degree-academic program",
    "Bachelors degree(BA AB BS)",
    "Masters degree(MA MS MEng MEd MSW MBA)",
    "Prof school degree (MD DDS DVM LLB JD)",
    "Doctorate degree(PhD EdD)",
]
census["EDUCATION"] = census["EDUCATION"].astype(pd.CategoricalDtype(BILDUNG, ordered=True))
# NUM_EMPLOYEES sind Grössenklassen 0 bis 6, also ordinal und keine Anzahl (4.1)
census["NUM_EMPLOYEES"] = census["NUM_EMPLOYEES"].astype(pd.CategoricalDtype(list(range(7)), ordered=True))
# Nominale Codes, die als Zahl gespeichert sind (1 und 4.1)
nominale_codes = ["INDUSTRY_CODE", "OCCUPATION_CODE", "SELF_EMPLOYED", "VETERANS_BENEFITS"]
census[nominale_codes] = census[nominale_codes].astype("category")

# Kontrolle: Eine Bildungskategorie, die nicht in BILDUNG steht, würde still zu NaN werden
assert census["EDUCATION"].notna().all(), "Unbekannte Kategorie in EDUCATION"

print(f"census (bereinigt): {census.shape[0]} Zeilen, {census.shape[1]} Spalten")
print(f"census_raw:         {census_raw.shape[0]} Zeilen, {census_raw.shape[1]} Spalten\n")
fehlend = census.isna().sum()
print("Fehlende Werte nach der Bereinigung:")
print(fehlend[fehlend > 0].sort_values(ascending=False))